# Step 20.4 — Resume Training

本节范围：reconstruct model、restore state、continue from checkpoint。20.3 实际提交 `666245d76848938cf3c3545e9d4376beaaf1d254` 已独立核心静态验收 PASS，累计修正 1/2；20.2 累计 1/2，19.5 为 2/2。这是材料验收，不是学员掌握或运行反馈。

中断后首次完整交付修正计 1/2。所有代码单元均未执行、无输出；不读取语料、不保存/加载真实 checkpoint、不恢复训练。完成即停在 20.4，不进入 20.5。

## 1. 恢复需要配对哪些对象？

**Resume training（恢复训练）**不是单独加载模型权重。未来要继续优化，至少要将模型权重（20.1）、Adam 状态树（20.2）和架构/tokenizer/进度元数据（20.3）配成同一次 run 的一组产物。

即使权重、Adam state 和 metadata 都匹配，如果没有完整 RNG state、sampler cursor、数据状态或相容的软件/硬件环境，也不能声称精确复现每一个后续随机选择。可以在参数与优化器状态匹配的前提下继续优化，但要区分 **continue optimization** 和 **exact trajectory reproduction**。

In [ ]:
import hashlib
import json
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten, tree_map, tree_unflatten

# Explicit location only; never auto-select a "latest" checkpoint.
CHECKPOINT_DIR = Path("results/checkpoints/REPLACE_WITH_REAL_RUN_DIRECTORY")
METADATA_NAME = "run_metadata.json"
WEIGHTS_NAME = "model.safetensors"
OPTIMIZER_ARRAYS_NAME = "optimizer_state.npz"
OPTIMIZER_MANIFEST_NAME = "optimizer_state.json"

# Must match the original run; values here are course candidates, not discovered metadata.
ADAM_CONFIG = {
    "learning_rate": 3e-4,
    "betas": [0.9, 0.999],
    "eps": 1e-8,
    "bias_correction": True,  # Explicit course choice; MLX v0.32.2 default is False.
}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def require_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Required checkpoint artifact is missing: {path.resolve()}")
    return path

## 2. 完整 MiniGPT 与架构恢复

先读 metadata，再按记录的 `dims`、`num_heads`、`num_blocks`、`max_seq_len` 和 `vocab_size` 构造同样的模型。沿用已验收的 Step 15.4 完整模型定义，不使用 toy 模型，也不从缺失文件猜配置。

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


## 3. 读取 Metadata 并重建 Tokenizer

20.3 保存了完整 `stoi`、`itos_by_id` 和 SHA-256 指纹，因此可以从 metadata 重建映射，而无需读取语料。恢复前核对词表大小、映射一致性与 fingerprint；不能只比较 vocab_size。

In [ ]:
def reconstruct_stoi(tokenizer):
    if tokenizer.get("type") != "character":
        raise ValueError("Unsupported tokenizer type.")
    chars = tokenizer.get("itos_by_id")
    if not isinstance(chars, list) or len(chars) != tokenizer.get("vocab_size"):
        raise ValueError("Invalid saved character table.")
    if any(not isinstance(ch, str) or len(ch) != 1 for ch in chars):
        raise ValueError("Invalid character entry.")
    stoi = {ch: i for i, ch in enumerate(chars)}
    if len(stoi) != len(chars) or tokenizer.get("stoi") != stoi:
        raise ValueError("Tokenizer mapping mismatch.")
    canonical = json.dumps([[ch, i] for i, ch in enumerate(chars)],
                           ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    if hashlib.sha256(canonical).hexdigest() != tokenizer.get("stoi_sha256"):
        raise ValueError("Tokenizer fingerprint mismatch.")
    return stoi

def load_checkpoint_metadata(checkpoint_dir):
    root = Path(checkpoint_dir).resolve()
    metadata_path = require_file(root / METADATA_NAME)
    with metadata_path.open("r", encoding="utf-8") as stream:
        metadata = json.load(stream)
    if metadata.get("format_version") != 1:
        raise ValueError("Unsupported metadata format version.")
    architecture = metadata.get("architecture")
    required = ("dims", "num_heads", "num_blocks", "max_seq_len", "vocab_size")
    if not isinstance(architecture, dict) or any(k not in architecture for k in required):
        raise ValueError("Architecture metadata is incomplete.")
    stoi = reconstruct_stoi(metadata.get("tokenizer", {}))
    if len(stoi) != architecture["vocab_size"]:
        raise ValueError("Architecture vocab_size and tokenizer mapping disagree.")
    return root, metadata, stoi

## 4. 校验 Artifact 引用

从 metadata 明确读取权重、优化器数组和优化器 manifest 的相对路径、大小与 SHA-256。相对路径必须留在指定 checkpoint 目录内。哈希匹配只能证明文件内容与记录一致，不能单独证明三份文件来自同一训练过程；还需核对 run 的进度、架构和优化器状态。

In [ ]:
def load_checkpoint_metadata(metadata_path):
    metadata_path = require_file(metadata_path).resolve()
    root = metadata_path.parent
    with metadata_path.open("r", encoding="utf-8") as stream:
        metadata = json.load(stream)
    if metadata.get("format_version") != 1:
        raise ValueError("Unsupported metadata format version.")
    architecture = metadata.get("architecture")
    required = ("dims", "num_heads", "num_blocks", "max_seq_len", "vocab_size")
    if not isinstance(architecture, dict) or any(k not in architecture for k in required):
        raise ValueError("Architecture metadata is incomplete.")
    stoi = reconstruct_stoi(metadata.get("tokenizer", {}))
    if len(stoi) != architecture["vocab_size"]:
        raise ValueError("Architecture vocab_size and tokenizer mapping disagree.")
    return metadata_path, metadata, dict(stoi)

def resolve_artifact(artifact_root, record, label):
    if not isinstance(record, dict) or not isinstance(record.get("relative_path"), str):
        raise ValueError(f"Missing artifact metadata for {label}.")
    root = Path(artifact_root).resolve()
    path = (root / record["relative_path"]).resolve()
    try:
        path.relative_to(root)
    except ValueError as exc:
        raise ValueError(f"Artifact path escapes artifact root: {label}") from exc
    require_file(path)
    if path.stat().st_size != record.get("size_bytes"):
        raise ValueError(f"Artifact size mismatch: {label}")
    if sha256_file(path) != record.get("sha256"):
        raise ValueError(f"Artifact SHA-256 mismatch: {label}")
    return path

def resolve_checkpoint_artifacts(artifact_root, metadata):
    # artifact_root is the SAME root passed to 20.3 artifact_metadata(...),
    # not necessarily the directory containing run_metadata.json.
    artifacts = metadata.get("artifacts", {})
    return {
        "weights": resolve_artifact(artifact_root, artifacts.get("weights"), "weights"),
        "optimizer_arrays": resolve_artifact(artifact_root, artifacts.get("optimizer_arrays"), "optimizer_arrays"),
        "optimizer_manifest": resolve_artifact(artifact_root, artifacts.get("optimizer_manifest"), "optimizer_manifest"),
    }

## 5. 重建模型并严格加载权重

固定 MLX v0.32.2 的 `Module.load_weights(file, strict=True)` 用于加载权重。恢复时先构造与 metadata 一致的完整模型，再严格加载指定文件；shape/参数路径不匹配应停止，不要通过放宽 strict 掩盖错误。

In [ ]:
def reconstruct_model(metadata):
    cfg = metadata["architecture"]
    return MiniGPT(
        vocab_size=cfg["vocab_size"],
        dims=cfg["dims"],
        num_heads=cfg["num_heads"],
        num_blocks=cfg["num_blocks"],
        max_seq_len=cfg["max_seq_len"],
    )

def load_model_weights_strict(model, weights_path):
    require_file(weights_path)
    model.load_weights(str(weights_path), strict=True)
    mx.eval(model.parameters())
    return model

In [ ]:
def loss_fn(model, X, Y):
    # X/Y are (B,T); Y is already shifted by the sampler.
    logits = model(X)  # (B,T,V)
    B, T, V = logits.shape
    flat_logits = mx.reshape(logits, (B * T, V))
    flat_targets = mx.reshape(Y, (B * T,))
    per_token = nn.losses.cross_entropy(flat_logits, flat_targets, axis=-1, reduction="none")
    return mx.mean(per_token)  # scalar MLX array

def global_grad_norm(grad_tree):
    leaves = tree_flatten(grad_tree)
    squared_sums = [mx.sum(mx.square(leaf)) for _, leaf in leaves]
    if not squared_sums:
        raise ValueError("Gradient tree has no array leaves.")
    return mx.sqrt(sum(squared_sums))

def clip_grad_tree(grad_tree, max_norm, eps_clip=1e-12):
    if max_norm <= 0:
        raise ValueError("clip_max_norm must be positive.")
    norm = global_grad_norm(grad_tree)
    mx.eval(norm)
    scale = mx.minimum(
        mx.array(1.0, dtype=mx.float32),
        mx.array(max_norm, dtype=mx.float32) / (norm + eps_clip),
    )
    return tree_map(lambda grad: grad * scale, grad_tree), norm, scale

## 6. 重建 Adam 并恢复状态

必须先用原运行的 Adam 超参数创建 optimizer，并针对当前模型的 trainable parameters 初始化 state tree，再加载 20.2 的状态。当前 learning_rate 数组快照不是调度器；若原训练使用 scheduler，必须单独重建相同规则与进度。

下列 loader 沿用 20.2 已发布的状态格式，先校验路径、数组身份、shape、dtype 与非数组类型，全部通过后才赋值。

In [ ]:
def restore_optimizer_state_from_files(optimizer, arrays_path, manifest_path):
    with Path(manifest_path).open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    if manifest.get("format_version") != 2:
        raise ValueError("Unsupported optimizer-state format version.")
    if manifest.get("optimizer_class") != type(optimizer).__name__:
        raise ValueError("Optimizer class does not match saved state.")

    archive = mx.load(str(arrays_path))
    path_to_value = {}
    for item in manifest["arrays"]:
        value = archive[item["key"]]
        if list(value.shape) != item["shape"] or str(value.dtype) != item["dtype"]:
            raise ValueError(f"Saved array metadata mismatch at {item['path']}")
        path_to_value[item["path"]] = value
    for item in manifest["python_scalars"]:
        value = item["value"]
        expected_type = item["python_type"]
        if expected_type == "NoneType":
            if value is not None:
                raise ValueError(f"Scalar type mismatch at {item['path']}")
        elif type(value).__name__ != expected_type:
            raise ValueError(f"Scalar type mismatch at {item['path']}")
        path_to_value[item["path"]] = value

    # Validate against the target optimizer BEFORE mutating optimizer.state.
    current_flat = tree_flatten(optimizer.state)
    current_paths = [str(path) for path, _ in current_flat]
    if set(current_paths) != set(path_to_value):
        raise ValueError("Saved optimizer-state paths do not match the current optimizer.")
    rebuilt_items = []
    for path, current_value in current_flat:
        path_text = str(path)
        saved_value = path_to_value[path_text]
        current_is_array = isinstance(current_value, mx.array)
        saved_is_array = isinstance(saved_value, mx.array)
        if current_is_array != saved_is_array:
            raise ValueError(f"Array/non-array identity mismatch at {path_text}")
        if current_is_array:
            if tuple(current_value.shape) != tuple(saved_value.shape):
                raise ValueError(f"Shape mismatch at {path_text}: {current_value.shape} vs {saved_value.shape}")
            if current_value.dtype != saved_value.dtype:
                raise ValueError(f"Dtype mismatch at {path_text}: {current_value.dtype} vs {saved_value.dtype}")
        elif type(current_value) is not type(saved_value):
            raise ValueError(f"Non-array leaf type mismatch at {path_text}")
        rebuilt_items.append((path, saved_value))

    # Only assign after every path/type/shape/dtype check has passed.
    rebuilt = tree_unflatten(rebuilt_items)
    optimizer.state = rebuilt
    mx.eval(optimizer.state)

# Definition only. No state is loaded during material preparation.

In [ ]:
def create_model_and_optimizer(metadata):
    model = reconstruct_model(metadata)
    optimizer = optim.Adam(**ADAM_CONFIG)
    optimizer.init(model.trainable_parameters())
    return model, optimizer

# Future work-copy orchestration; select both locations explicitly.
# metadata_path = Path(".../run_metadata/<run-id>/run_metadata.json")
# artifact_root = Path(".../the-original-artifact-root-used-by-20.3")
# metadata_path, metadata, stoi = load_checkpoint_metadata(metadata_path)
# artifacts = resolve_checkpoint_artifacts(artifact_root, metadata)
# model, optimizer = create_model_and_optimizer(metadata)
# load_model_weights_strict(model, artifacts["weights"])
# restore_optimizer_state_from_files(optimizer, artifacts["optimizer_arrays"], artifacts["optimizer_manifest"])
# verify_completed_update_count(optimizer, metadata)
# train_step = make_resumed_train_step(model, optimizer)
# progress_log = resume_update_loop(
#     model, optimizer, train_step, explicit_train_tokens,
#     completed_updates=metadata["training_progress"]["completed_updates"],
#     target_total_updates=desired_total_updates,
#     batch_size=chosen_batch_size, block_size=metadata["architecture"]["max_seq_len"],
#     train_rng=explicit_restored_rng,
# )

## 7. 核对进度与 Adam step

20.3 的 `training_progress.completed_updates` 应与恢复的 Adam step 一致。固定 MLX v0.32.2 的 Adam step 保存在 optimizer state tree 中，通常是零维 uint64 MLX 数组；比较前显式求值，再读取整数。不要把初始 step=0 的新 optimizer 误认为已恢复状态。

由于状态树内部布局属于具体 optimizer 实现，以下辅助函数递归寻找名为 `step` 的叶子；若不存在或不唯一则明确停止，而不是猜测。

In [ ]:
def find_named_step_leaf(tree):
    matches = []
    def walk(node, path=()):
        if isinstance(node, dict):
            for key, value in node.items():
                next_path = path + (str(key),)
                if str(key) == "step":
                    matches.append((next_path, value))
                walk(value, next_path)
        elif isinstance(node, (list, tuple)):
            for index, value in enumerate(node):
                walk(value, path + (str(index),))
    walk(tree)
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one Adam step leaf, found {len(matches)}.")
    return matches[0]

def verify_completed_update_count(optimizer, metadata):
    progress = metadata.get("training_progress", {})
    expected = progress.get("completed_updates")
    if type(expected) is not int or expected < 0:
        raise ValueError("Metadata must contain a real non-negative completed_updates value.")
    path, step_value = find_named_step_leaf(optimizer.state)
    if not isinstance(step_value, mx.array):
        raise TypeError(f"Adam step at {path} is not an MLX array.")
    mx.eval(step_value)
    actual = int(step_value.item())
    if actual != expected:
        raise ValueError(f"Adam step {actual} does not match metadata completed_updates {expected}.")
    return actual

# Call only after restoring the optimizer state in a future work copy.

## 8. 继续优化与严格复现的区别

恢复成功意味着模型权重、优化器状态、架构、tokenizer 与进度等关键对象通过匹配检查，可以在这些条件下继续更新。它不自动保证后续训练与原运行逐步完全一致。

严格复现还可能要求完整 Python/MLX RNG state、训练 sampler 的内部状态/游标、数据版本和顺序、学习率调度器状态、软件版本、设备和一致的执行条件。20.3 只记录初始 seed，并默认 `rng_state_saved=False`；若没有真实保存的 RNG state，就不能声称精确轨迹续接。

## 9. 后续训练入口（定义协议，不执行）

恢复完成后，未来工作副本才可用真实训练 split 与 RNG 状态构造 batch，再调用与 19.5 相同的稳定性单步。必须从 metadata 中真实的 `completed_updates` 继续计数，而不是把新循环重新标成从 0 开始。

本节不自动读取语料或生成 sampler；调用者必须显式提供与 tokenizer 一致的 token stream，并核对 split 与原记录一致。

In [ ]:
def get_batch(data, batch_size, block_size, rng):
    if len(data.shape) != 1:
        raise ValueError("Expected a 1-D token stream.")
    if type(batch_size) is not int or batch_size < 1 or type(block_size) is not int or block_size < 1:
        raise ValueError("batch_size and block_size must be positive integers.")
    n = data.shape[0]
    if n < block_size + 1:
        raise ValueError("Split must contain at least block_size + 1 tokens.")
    starts = [rng.randrange(0, n - block_size) for _ in range(batch_size)]
    xs, ys = [], []
    for start in starts:
        chunk = data[start:start + block_size + 1]
        xs.append(chunk[:-1])
        ys.append(chunk[1:])
    return mx.stack(xs, axis=0), mx.stack(ys, axis=0), starts

In [ ]:
def make_resumed_train_step(model, optimizer, *, clip_max_norm=1.0):
    # Build a fresh gradient transform bound to THIS restored model, not an old global callable.
    loss_and_grad = nn.value_and_grad(model, loss_fn)

    def stable_step(X, Y):
        loss_before_update, grads = loss_and_grad(model, X, Y)
        grad_norm = global_grad_norm(grads)
        mx.eval(loss_before_update, grad_norm)
        if not bool(mx.all(mx.isfinite(loss_before_update)).item()):
            raise FloatingPointError("Training loss is NaN or Inf; optimizer not updated.")
        if not bool(mx.all(mx.isfinite(grad_norm)).item()):
            raise FloatingPointError("Gradient norm is NaN or Inf; optimizer not updated.")

        clipped_grads, preclip_norm, clip_scale = clip_grad_tree(grads, clip_max_norm)
        mx.eval(preclip_norm, clip_scale)
        if not bool(mx.all(mx.isfinite(preclip_norm)).item()):
            raise FloatingPointError("Pre-clipping gradient norm is NaN or Inf; optimizer not updated.")
        optimizer.update(model, clipped_grads)
        mx.eval(model.state, optimizer.state)
        return loss_before_update, grad_norm, preclip_norm, clip_scale

    return stable_step

def resume_update_loop(model, optimizer, train_step, train_tokens, *,
                       completed_updates, target_total_updates, batch_size,
                       block_size, train_rng):
    if type(completed_updates) is not int or completed_updates < 0:
        raise ValueError("completed_updates must be a non-negative integer from metadata.")
    if type(target_total_updates) is not int or target_total_updates < completed_updates:
        raise ValueError("target_total_updates must be >= completed_updates.")
    if train_tokens is None:
        raise ValueError("Supply the explicitly restored training split; this helper does not load corpus.")
    if train_tokens.shape[0] < block_size + 1:
        raise ValueError("Training split is too short for block_size+1 tokens.")

    # Check progress BEFORE sampling or updating; stale progress cannot restart a loop.
    step_path, step_value = find_named_step_leaf(optimizer.state)
    if not isinstance(step_value, mx.array):
        raise TypeError(f"Adam step at {step_path} is not an MLX array.")
    mx.eval(step_value)
    actual_step = int(step_value.item())
    if actual_step != completed_updates:
        raise ValueError(f"Adam step {actual_step} does not match supplied completed_updates {completed_updates}.")

    progress_log = []
    for _ in range(completed_updates, target_total_updates):
        before_path, before_value = find_named_step_leaf(optimizer.state)
        mx.eval(before_value)
        before_step = int(before_value.item())
        if before_step != completed_updates:
            raise ValueError(f"Progress changed unexpectedly: expected {completed_updates}, got {before_step}.")

        X, Y, _ = get_batch(train_tokens, batch_size, block_size, train_rng)
        mx.eval(X, Y)
        loss_before_update, grad_norm, preclip_norm, clip_scale = train_step(X, Y)

        after_path, after_value = find_named_step_leaf(optimizer.state)
        mx.eval(loss_before_update, after_value, model.state, optimizer.state)
        after_step = int(after_value.item())
        if after_step != before_step + 1:
            raise RuntimeError(f"Optimizer step must advance exactly once: {before_step} -> {after_step}.")
        completed_updates = after_step
        progress_log.append({
            "completed_updates": completed_updates,
            "training_loss_pre_update": float(loss_before_update.item()),
            "gradient_norm_preclip": float(grad_norm.item()),
            "gradient_norm_used_for_clipping": float(preclip_norm.item()),
            "clip_scale": float(clip_scale.item()),
        })
    return progress_log

# Future work-copy orchestration (not called in this master):
# metadata_path = Path(".../run_metadata.json")  # explicitly selected
# artifact_root = Path(".../the_original_artifact_root")  # root used by 20.3 artifact_metadata
# metadata_path, metadata, stoi = load_checkpoint_metadata(metadata_path)
# artifacts = resolve_checkpoint_artifacts(artifact_root, metadata)
# model = reconstruct_model(metadata)
# optimizer = optim.Adam(**ADAM_CONFIG); optimizer.init(model.trainable_parameters())
# load_model_weights_strict(model, artifacts["weights"])
# restore_optimizer_state_from_files(optimizer, artifacts["optimizer_arrays"], artifacts["optimizer_manifest"])
# verify_completed_update_count(optimizer, metadata)
# train_step = make_resumed_train_step(model, optimizer)
# progress_log = resume_update_loop(model, optimizer, train_step, explicit_train_tokens, ...)

## 本节边界与累计计数

本节定义恢复顺序与检查：读取 metadata → 重建 tokenizer/architecture → 校验 artifact → 构造模型与 Adam → 严格加载权重 → 恢复 optimizer state → 核对 step/progress → 由调用者显式提供数据/RNG 后继续优化。

没有完整 RNG/sampler/数据状态时，不宣称精确复现轨迹。真实文件缺失、hash/shape/dtype/架构/进度不匹配时应停止，不自动挑选“最新”文件、不猜测配置。

20.4 中断后首次完整交付修正 1/2；20.3 累计 1/2；20.2 累计 1/2；19.5 为 2/2；19.2/19.3 各 1/2；其余历史计数保留。所有代码单元未执行、无输出。本轮未读取语料、未保存/加载真实 checkpoint、未恢复训练。完成即停在20.4，不进入20.5；大Step20全部材料完成后停止，不进入21。